In [1]:
# load dataset
%env HF_DATASETS_OFFLINE=1

from datasets import load_dataset

data_hf = load_dataset("deepcopy/NIAID-Molecule-OCR-Real-Images-Dataset")

env: HF_DATASETS_OFFLINE=1


Using the latest cached version of the dataset since deepcopy/NIAID-Molecule-OCR-Real-Images-Dataset couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'default' at /root/.cache/huggingface/datasets/deepcopy___niaid-molecule-ocr-real-images-dataset/default/0.0.0/47b3fa7216fa339e8692c621e1bbe51f80a28d0c (last modified on Mon Jul 20 21:49:22 2026).


In [2]:
# split
data_split_hf = data_hf['train'].train_test_split(test_size=0.2, seed=42)
train_hf, test_hf = data_split_hf['train'], data_split_hf['test']

In [3]:
from PIL import Image
from rdkit import Chem
from rdkit.Chem import Draw

def replace_image(batch):
    handwritten_images: list[Image.Image] = batch["image"]
    smiles: list[str] = batch["smiles"]
    new_images = []
    for idx, smi in enumerate(smiles):
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            try:
                img = Draw.MolToImage(mol, size=(384, 384))
                new_images.append(img)
                continue
            except Exception as exc:
                print(f"WARNING: Failed to draw SMILES string '{smi}' at index {idx}: {exc}. Using handwritten image instead.")
        else:
            print(f"WARNING: Invalid SMILES string '{smi}' at index {idx}. Using handwritten image instead.")
        new_images.append(handwritten_images[idx])

    return {
        "handwritten_image": handwritten_images,
        "image": new_images,
        "smiles": smiles,
    }

test_hf = test_hf.map(replace_image, batched=True, batch_size=16, remove_columns=test_hf.column_names)

In [4]:
# load vlm
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-4B-Instruct", 
    dtype="auto", 
    device_map="auto",
    local_files_only=True,
)

processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen3-VL-4B-Instruct",
    local_files_only=True,
    processor_kwargs={
        "padding_side": "left",
    },
)

print("LOG: Model and processor loaded successfully.")

# def generate_qwen3vl(image, prompt) -> str:
#     messages = [
#         {
#             "role": "user",
#             "content": [
#                 {
#                     "type": "image",
#                     "image": image,
#                 },
#                 {"type": "text", "text": prompt},
#             ],
#         }
#     ]

#     inputs = processor.apply_chat_template(
#         messages,
#         tokenize=True,
#         add_generation_prompt=True,
#         return_dict=True,
#         return_tensors="pt"
#     )
#     inputs = inputs.to(model.device)

#     # Inference: Generation of the output
#     generated_ids = model.generate(**inputs, max_new_tokens=256)
#     generated_ids_trimmed = [
#         out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
#     ]
#     output_text = processor.batch_decode(
#         generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
#     )
#     return output_text[0]

def batch_generate_qwen3vl(processor, model, image_list: list, prompt_list: list[str]) -> list[str]:
    assert len(image_list) == len(prompt_list), "The number of images and prompts must be the same."
    messages = [
        [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": img,
                    },
                    {"type": "text", "text": prmpt},
                ],
            }
        ] for img, prmpt in zip(image_list, prompt_list)
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={"padding": True},
    )
    inputs = inputs.to(model.device)

    # Inference: Generation of the output
    generated_ids = model.generate(**inputs, max_new_tokens=256)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text

BATCH_SIZE = 4
prompt = "Generate the SMILES string for the molecule in the image. Do not output anything else."

from pathlib import Path
import jsonlines
from datetime import datetime
from tqdm import tqdm

def generate_ocr_qwenvl3(data, prompt, batch_size, processor, model):
    # prepare output file
    output_dir = Path("data/exp")
    output_dir.parent.mkdir(parents=True, exist_ok=True)
    date_time = datetime.now().strftime("%Y%m%d-%H%M%S")
    run_name = f"NIAID-Qwen3VL-4B-{date_time}.jsonl"
    output_file = output_dir / run_name
    output_dir.mkdir(parents=True, exist_ok=True)

    # TODO: save rdkit and original images
    image_save_dir = output_dir / f"NIAID-images-RDKit" 
    image_save_dir.mkdir(parents=True, exist_ok=True)

    global_idx = 0

    for start in tqdm(range(0, len(test_hf), batch_size)):
        batch = data.select(range(start, min(start + batch_size, len(test_hf))))
        handwritten_image_list = batch["handwritten_image"]
        image_list = batch["image"]
        prompt_list = [prompt] * len(image_list)
        gt_list = batch["smiles"]

        output_texts = batch_generate_qwen3vl(processor, model, image_list, prompt_list)
        for i, output_text in enumerate(output_texts):
            # save image
            cur_image_save_dir = image_save_dir / f"{global_idx:06d}"
            cur_image_save_dir.mkdir(parents=True, exist_ok=True)
            handwritten_image_path = cur_image_save_dir / "handwritten_image.png"
            handwritten_image_list[i].save(handwritten_image_path)
            image_path = cur_image_save_dir / "rdkit_image.png"
            image_list[i].save(image_path)

            # save output
            with jsonlines.open(output_file, mode='a') as writer:
                writer.write({
                    "index": f"{global_idx:06d}",
                    "handwritten_image_path": str(handwritten_image_path),
                    "image_path": str(image_path),
                    "prompt": prompt,
                    "ground_truth": gt_list[i],
                    "output_text": output_text
                })
            global_idx += 1


generate_ocr_qwenvl3(test_hf, prompt, BATCH_SIZE, processor, model)            

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

LOG: Model and processor loaded successfully.


  1%|          | 3/255 [00:13<19:01,  4.53s/it]


KeyboardInterrupt: 